# MethaneScope AI — Lesson 2.5
## Reusable satellite ingestion application

**Starting point:** Lesson 2.3 accepted a real NASA EMIT granule and Lesson 2.4 inserted it into PostGIS. We will **not** delete that data.

**Objectives:** run file discovery, repeat QA checks, execute idempotent database ingestion, read logs and run automated tests.

**Boundary:** An ACCEPTED engineering record is not independent scientific validation of methane enhancements or emissions rates.

### A. Our architecture

`data/raw/nasa_emit/` → `discover` (inventory only) → Lesson 2.3 acceptance manifest → `validate` (independent checks) → `ingest` (single-granule transaction) → PostgreSQL/PostGIS → CSV/JSON audit + logs.

The difference between **file discovery** and **scientific acceptance** is important: finding a file is not proof that it is fit for analysis.

In [ ]:
from pathlib import Path
import sys
import subprocess
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "methanescope" / "pipeline.py").exists():
    raise FileNotFoundError("Open JupyterLab from MethaneScope root and extract the Lesson 2.5 ZIP there.")

print("Project root:", ROOT)
print("Python interpreter:", sys.executable)
print("Accepted CSV present:", (ROOT / "outputs/emit_batch/emit_accepted_for_ingestion.csv").is_file())

### B. Install dependencies (only if needed)

If any import fails, run this cell once and **restart the kernel**. Do not install into a different Python interpreter. Your `Python (MethaneScope)` kernel is recommended.

In [ ]:
# Uncomment if dependencies are missing:
# %pip install -r requirements-lesson25.txt

from methanescope.science import discover, load_manifest, validate_one
print("MethaneScope Python module imported successfully")

### C. Discover NASA input pairs

Look for `CH4PLM` / `CH4PLMMETA`. This command **only inventories** candidates.

In [ ]:
pairs = discover(ROOT)
display(pd.DataFrame(pairs))
print("Candidates found:", len(pairs))

### D. Inspect the manifest from Lesson 2.3

In [ ]:
mode, rows = load_manifest(ROOT)
print("Mode:", mode)
print("ACCEPTED rows:", len(rows))
display(pd.DataFrame(rows)[["granule_id", "status", "path", "geojson_path"]])

### E. Revalidate accepted files before connecting to the database

Checks include: source paths remain inside the project, both checksums match, filenames match the NASA asset pairing scheme, raster/CRS/pixel values match, and GeoJSON geometry is valid.

In [ ]:
validated = [validate_one(ROOT, row) for row in rows]
for item in validated:
    print("VALID:", item.manifest["granule_id"])
    print("  Extent WGS84:", item.bounds_wgs84)
    print("  GeoJSON features:", len(item.features))

### F. Run the same validation through the reusable CLI

It should print `VALID` and write an audit. If not, **do not proceed to ingestion**.

In [ ]:
def run_command(command):
    result = subprocess.run(
        [sys.executable, "-m", "methanescope.pipeline", command, "--root", str(ROOT)],
        cwd=ROOT, capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print("stderr:", result.stderr)
    print("Exit code:", result.returncode)
    return result.returncode

assert run_command("validate") == 0, "Validation failed; inspect output before ingesting"

### G. Ingest the already-approved granule into PostGIS

**Expected for your existing database:** `SKIPPED`, because Lesson 2.4 inserted the observation already. That confirms duplicate prevention. Ensure Docker Desktop and the PostGIS container are running.

If the connection fails, inspect the error, check `docker compose -f docker-compose.postgis.yml ps`, and rerun. Never delete the database volume.

In [ ]:
ingest_exit_code = run_command("ingest")
if ingest_exit_code:
    print("Ingestion needs attention; inspect outputs/pipeline/pipeline.log")

### H. Query source counts without directly editing the database

In [ ]:
status_exit_code = run_command("status")
if status_exit_code:
    print("Check Docker container status and .env settings")

### I. Inspect the latest audit report

In [ ]:
output_dir = ROOT / "outputs/pipeline"
reports = sorted(output_dir.glob("*_ingest_summary.json"))
if reports:
    report = json.loads(reports[-1].read_text(encoding="utf-8"))
    print(json.dumps(report, indent=2))
else:
    print("No ingest summary found yet. Run the ingest cell above.")

### J. Run automated unit tests

These tests use generated mock data, not real methane measurement claims. They check filename matching, checksums, safe file paths and manifest assumptions. No database reset is needed.

In [ ]:
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_science.py"],
    cwd=ROOT, capture_output=True, text=True
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr)
print("Test exit code:", test_result.returncode)

### Knowledge checkpoint

1. Why does `discover` not automatically mean `ACCEPTED`?
2. Why is `SKIPPED` a good result after Lesson 2.4 already inserted the granule?
3. Why can't we change the original NASA `.tif` or `.json` files without rerunning QA?
4. What are the differences between `REVIEW`, `CONFLICT`, `ERROR`, and `SKIPPED`?
5. What further scientific validation is required before using these observations to estimate methane emissions?

**Next:** operational monitoring and scheduled processing (Lesson 2.6).